# 72. 원판 검출에서 AdamW 초기 학습률과 감소 방식 비교

최대 200에폭·patience 30을 고정합니다. 초기 학습률(lr0) 0.001과 0.0005, 선형 감소와 코사인 감소를 조합해 네 모델을 같은 RTX 3090에서 원본 가중치로 각각 학습합니다. 기존 자동 설정이 실제로 선택했던 AdamW와 β1=0.9, bias의 워밍업 시작 학습률 0을 네 조건에서 동일하게 유지합니다.

아래 첫 표에서 각 조건이 실제로 몇 에폭까지 진행됐는지 확인합니다. 이전 200/30(실제 lr0 0.002, 선형)은 별도 호스트에서 학습한 참고 모델입니다.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT
from wlcv.training.plate_lr_grid import CASES

experiment = ROOT / "outputs/plate_lr_grid/20261126"
review = experiment / "review"
audits = {}
histories = {}
for name in CASES:
    run = experiment / "runs" / name
    audits[name] = json.loads((run / "audit.json").read_text())
    histories[name] = pd.read_csv(run / "results.csv").set_index("epoch")
display(pd.DataFrame(audits).T[["initial_lr", "cos_lr", "best_epoch", "epochs_run", "train_and_val_seconds"]])

## 설정한 학습률이 실제로 적용됐나

학습 로그의 파라미터 그룹 0에서 기록한 학습률을 그립니다. 검증 손실과 mAP는 큰 흐름을 볼 수 있도록 최근 5에폭 평균으로 표시합니다. 조기 종료된 실행은 마지막으로 측정한 에폭까지만 이어집니다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
colors = {0.001: "#21628b", 0.0005: "#b45a23"}
for ax, metric, title in zip(axes.flat,
        ["lr/pg0", "val/box_loss", "val/cls_loss", "metrics/mAP50-95(B)"],
        ["Recorded learning rate (group 0)", "Validation box loss", "Validation class loss", "Validation mAP50-95"]):
    for name, (lr, cosine) in CASES.items():
        values = histories[name][metric]
        if metric != "lr/pg0":
            values = values.rolling(5, min_periods=1).mean()
        ax.plot(values, color=colors[lr], linestyle="--" if cosine else "-",
                label=f"{lr} | {'cosine' if cosine else 'linear'}")
    ax.set(title=title, xlabel="Epoch")
    ax.grid(alpha=0.2)
axes[0, 0].legend(fontsize=8)
fig.savefig(review / "training_curves.png", dpi=145)
plt.show()

## 어떤 조건에서 검출과 허브 상승량이 좋아졌나

미탐은 수동 박스가 있는 동작 중 270장에서 셉니다. 좌표 누락은 시간 연결 후에도 박스 중심이 없는 동작 중 프레임(전체 478장)입니다. 상승량은 무릎 통과와 접촉 프레임의 y좌표 차이입니다. 공통으로 측정할 수 있는 선수에서 수동 상승량과의 절대오차를 평균 냅니다. 각 선수의 부호 있는 오차에서 양수는 과대, 음수는 과소 측정입니다.

In [ ]:
summary = pd.read_csv(review / "summary.csv").set_index("case")
people = pd.read_csv(review / "athletes.csv")
display(summary.round(4))
errors = people.pivot(index="video_id", columns="case", values="rise_error_px")
display(errors.reindex(columns=summary.index).round(2))

## 동작 중 오탐과 준비·종료 후 오탐

박스를 출력했지만 수동 원판 박스와 IoU 0.50 기준을 충족하지 못한 프레임 수입니다. 수동 박스가 있는 프레임만 채점합니다. scored가 0인 구간은 평가 자료가 없습니다.

In [ ]:
fp = pd.read_csv(review / "false_positives.csv")
display(fp.pivot(index="case", columns="phase", values=["fp_frames", "scored"]).reindex(summary.index))

## 설정 한 가지씩 바꾸어 읽기

아래 표의 각 차이는 after에서 before를 뺀 값입니다. 같은 학습률에서 스케줄 효과를, 같은 스케줄에서 학습률을 절반으로 줄인 효과를 읽습니다. mAP는 증가가 개선이고 미탐·누락·오차는 감소가 개선입니다.

In [ ]:
display(pd.read_csv(review / "contrasts.csv").round(4))

## 대표 프레임 검수

b02의 바와 몸이 접촉하는 36번에서 원판 바깥 박스와 허브를 봅니다. 초록 점선은 수동 박스, 주황 실선은 선택된 박스, 흰색 십자는 직접 찍은 허브입니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import ANNOTATIONS, VIDEOS
from wlcv.data.frames import read_rgb

video, index = "b02_sn_h_02", 36
frames = pd.read_csv(review / "frames.csv")
rows = frames[(frames.video_id == video) & (frames.frame_index == index)].set_index("case")
hubs = pd.read_csv(ANNOTATIONS / "plate_blur_review/validation.csv")
hub = hubs[(hubs.video_id == video) & (hubs.frame_index == index)].iloc[0]
rgb = read_rgb(VIDEOS / f"{video}.mp4", index)
fig, axes = plt.subplots(1, len(CASES), figsize=(12, 3.4), layout="constrained")
for ax, name in zip(axes, CASES):
    row = rows.loc[name]
    ax.imshow(rgb)
    x, y = row.gt_x1, row.gt_y1
    w, h = row.gt_x2 - x, row.gt_y2 - y
    ax.add_patch(Rectangle((x,y), w, h, fill=False, edgecolor="lime", linestyle="--", linewidth=2))
    if pd.notna(row.x1):
        ax.add_patch(Rectangle((row.x1,row.y1),row.x2-row.x1,row.y2-row.y1,
                               fill=False,edgecolor="darkorange",linewidth=2))
    ax.plot(hub.manual_x,hub.manual_y,"w+",markersize=10,markeredgewidth=2)
    ax.set(xlim=(max(0,x-.3*w),min(rgb.shape[1],x+1.3*w)),
           ylim=(min(rgb.shape[0],y+1.3*h),max(0,y-.3*h)), title=f"{name}\nIoU {row.iou:.2f}")
    ax.axis("off")
fig.savefig(review / "b02_contact_local.png", dpi=140)
plt.show()